# Model Validation and Comparison Analysis

This tutorial demonstrates the systematic comparison of refractive index models at 1762 nm, validating the experimental results against established atmospheric models (Ciddor, Edlén, Mathar) and quantifying wavelength-specific differences.

## Overview

The analysis compares the experimentally determined refractive index model with three standard atmospheric models to:
1. Validate the experimental coefficients
2. Quantify wavelength-specific differences
3. Identify enhanced sensitivity near water absorption lines
4. Provide uncertainty estimates for model selection

## Code Structure

### 1. Initialization and Configuration

**Purpose**: Imports necessary libraries and model implementations. The color scheme ensures consistent, publication-quality visualizations.


In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

import sys, json
sys.path.append('..')

from models.nist.refractive_index import ciddor, edlen
from models.mathar.Mathar2007 import n

pub_gray = '#cecece'
pub_purple = '#a559aa'
pub_teal = '#59a89c'
pub_gold = '#f0c571'
pub_red = '#e02b35'
pub_blue = '#082a54'

colorlist = [pub_purple, pub_teal, pub_gold, pub_red, pub_blue]

### 2. Results Loading Function

**Purpose**: Loads the experimental regression coefficients for comparison with theoretical models.

In [2]:
def load_regression_results(filename='regression_results.json'):
    """
    Load regression analysis results from a JSON file.
    
    Parameters
    ----------
    filename : str, optional
        Path to the JSON file containing regression results (default: 'regression_results.json')
    
    Returns
    -------
    dict
        Dictionary containing regression coefficients, standard errors, and other results
    
    Notes
    -----
    The JSON file should contain the output from regression analysis including:
    - Coefficients (const, temperature, humidity, pressure)
    - Standard errors (OLS, HAC, Prais-Winsten)
    - Bootstrap confidence intervals
    - EIV systematic errors
    """
    with open(filename, 'r', encoding='utf-8') as f:
        results = json.load(f)
    print(f"Results loaded from {filename}")
    return results

### 3. Refractive Index Prediction Function

**Purpose**: Creates a callable function that predicts refractive index using the experimentally determined coefficients. This function serves as the "This Work" model for comparison.

In [3]:
def refractive_function(results_json):
    """Create prediction function from JSON coefficients"""
    coeff = results_json['coefficients']
    
    def n_1762(T, H, P):
        """
        Predict refractive index at 1762nm using coefficients from JSON
        
        Parameters:
        -----------
        T : float or array
            Temperature in °C
        H : float or array
            Humidity in %
        P : float or array
            Pressure in Pa
            
        Returns:
        --------
        float or array
            Predicted refractive index
        """
        return (coeff['const'] + 
                coeff['temperature'] * T + 
                coeff['humidity'] * H + 
                coeff['pressure'] * P)
    
    return n_1762

### 4. Temperature Dependence Comparison
**Purpose**: Compares temperature dependence across models at fixed humidity (30%) and pressure (99000 Pa). The analysis calculates and reports the temperature coefficient ($\alpha_T$) for each model.

### 5. Humidity Dependence Comparison

**Purpose**: Compares humidity dependence across models at fixed temperature (20°C) and pressure (99000 Pa). This analysis reveals the enhanced humidity sensitivity at 1762 nm.

### 6. Pressure Dependence Comparison

**Purpose**: Compares pressure dependence across models at fixed temperature (20°C) and humidity (30%). Pressure is displayed in hPa for readability.

In [11]:
# Load experimental results
results_json = load_regression_results('regression_results.json')
n_1762 = refractive_function(results_json)

# Create figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
plt.subplots_adjust(wspace=0.3)

# Temperature comparison (fixed humidity and pressure)
pressure1 = 99000
humidity1 = 30
temperature1 = np.linspace(15, 35, 1000)

models_temp = {
    'Ciddor@1700nm': np.array([ciddor(wave=1762, t=i, p=pressure1, rh=humidity1) for i in temperature1]),
    'Edlen@1700nm': np.array([edlen(wave=1762, t=i, p=pressure1, rh=humidity1) for i in temperature1]),
    'Mathar@1762nm': n(1.762, 273.15+temperature1, pressure1, humidity1),
    'This Work@1762nm': n_1762(temperature1, humidity1, pressure1)
}

# Plot temperature dependence
for data_info, color in zip(models_temp.items(), colorlist):
    label, values = data_info
    axes[0].plot(temperature1, values, color=color, label=label)
    
print("Temperature coefficient (dN/dT):")
for label, values in models_temp.items():
    slope, _ = np.polyfit(temperature1, values, 1)
    print(f"{label}: {slope:.3e}")

axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Refractive Index')
# axes[0].set_title('Temperature Comparison')
axes[0].grid(True)

pressure2 = 99000
humidity2 = np.linspace(20, 45, 1000)
temperature2 = 20

models_humidity = {
    'Ciddor@1700nm': np.array([ciddor(wave=1762, t=temperature2, p=pressure2, rh=i) for i in humidity2]),
    'Edlen@1700nm': np.array([edlen(wave=1762, t=temperature2, p=pressure2, rh=i) for i in humidity2]),
    'Mathar@1762nm': n(1.762, 273.15+temperature2, pressure2, humidity2),
    'This Work@1762nm': n_1762(temperature2, humidity2, pressure2)
}

for data_info, color in zip(models_humidity.items(), colorlist):
    label, values = data_info
    axes[1].plot(humidity2, values, color=color, label=label)
    
print("\nHumidity coefficient (dN/dH):")
for label, values in models_humidity.items():
    slope, _ = np.polyfit(humidity2, values, 1)
    print(f"{label}: {slope:.3e}")

axes[1].set_xlabel('Humidity (\%)')
# axes[1].set_title('Humidity Comparison')
axes[1].grid(True)

pressure3 = np.linspace(96000, 100000, 1000)
humidity3 = 30
temperature3 = 20

models_pressure = {
    'Ciddor@1700nm': np.array([ciddor(wave=1762, t=temperature3, p=i, rh=humidity3) for i in pressure3]),
    'Edlen@1700nm': np.array([edlen(wave=1762, t=temperature3, p=i, rh=humidity3) for i in pressure3]),
    'Mathar@1762nm': n(1.762, 273.15+temperature3, pressure3, humidity3),
    'This Work@1762nm': n_1762(temperature3, humidity3, pressure3)
}

for data_info, color in zip(models_pressure.items(), colorlist):
    label, values = data_info
    axes[2].plot(pressure3*1e-2, values, color=color, label=label)
    
print("\nPressure coefficient (dN/dP):")
for label, values in models_pressure.items():
    slope, _ = np.polyfit(pressure3, values, 1)
    print(f"{label}: {slope:.3e}")

axes[2].set_xlabel('Pressure (hPa)')
# axes[2].set_title('Pressure Comparison')
axes[2].grid(True)

for ax in axes:
    ax.legend()

plt.show()

Results loaded from regression_results.json


## Extended Analysis Functions

### 7. Model Deviation Quantification

**Purpose**: Quantifies differences between models in parts per million (ppm) to assess practical significance.


In [4]:
def calculate_model_deviations(models_temp, models_humidity, models_pressure):
    """
    Calculate percentage deviations between models
    
    Parameters:
    -----------
    models_temp : dict
        Temperature-dependent model predictions
    models_humidity : dict
        Humidity-dependent model predictions  
    models_pressure : dict
        Pressure-dependent model predictions
        
    Returns:
    --------
    dict
        Deviation statistics for each model comparison
    """
    deviations = {}
    
    # Use "This Work" as reference
    ref_temp = models_temp['This Work@1762nm']
    ref_humidity = models_humidity['This Work@1762nm']
    ref_pressure = models_pressure['This Work@1762nm']
    
    for model_name in ['Ciddor@1700nm', 'Edlen@1700nm', 'Mathar@1762nm']:
        # Temperature deviations
        temp_dev = (models_temp[model_name] - ref_temp) / ref_temp * 1e6  # ppm
        temp_rmse = np.sqrt(np.mean(temp_dev**2))
        
        # Humidity deviations
        humidity_dev = (models_humidity[model_name] - ref_humidity) / ref_humidity * 1e6
        humidity_rmse = np.sqrt(np.mean(humidity_dev**2))
        
        # Pressure deviations
        pressure_dev = (models_pressure[model_name] - ref_pressure) / ref_pressure * 1e6
        pressure_rmse = np.sqrt(np.mean(pressure_dev**2))
        
        deviations[model_name] = {
            'temperature_rmse_ppm': temp_rmse,
            'humidity_rmse_ppm': humidity_rmse,
            'pressure_rmse_ppm': pressure_rmse,
            'max_temp_dev_ppm': np.max(np.abs(temp_dev)),
            'max_humidity_dev_ppm': np.max(np.abs(humidity_dev)),
            'max_pressure_dev_ppm': np.max(np.abs(pressure_dev))
        }
    
    return deviations

### 8. Enhanced Sensitivity Analysis

**Purpose**: Quantifies the enhanced humidity sensitivity observed at 1762 nm relative to the Mathar model.

In [5]:
def analyze_enhanced_sensitivity(models_humidity):
    """
    Analyze enhanced humidity sensitivity at 1762 nm
    
    Parameters:
    -----------
    models_humidity : dict
        Humidity-dependent model predictions
        
    Returns:
    --------
    dict
        Enhanced sensitivity statistics
    """
    # Calculate slopes for humidity dependence
    humidity_range = np.linspace(20, 45, 1000)
    
    sensitivities = {}
    for label, values in models_humidity.items():
        slope, intercept = np.polyfit(humidity_range, values, 1)
        sensitivities[label] = slope
    
    # Calculate percentage enhancement relative to Mathar model
    mathar_slope = sensitivities['Mathar@1762nm']
    this_work_slope = sensitivities['This Work@1762nm']
    
    enhancement_percent = (this_work_slope - mathar_slope) / abs(mathar_slope) * 100
    
    return {
        'sensitivities': sensitivities,
        'enhancement_percent': enhancement_percent,
        'mathar_slope': mathar_slope,
        'this_work_slope': this_work_slope
    }

### 9. Wavelength Extrapolation Analysis
**Purpose**: Analyzes how model predictions vary with wavelength to understand spectral dependencies.

In [6]:
def analyze_wavelength_dependence():
    """
    Analyze model performance across different wavelengths
    """
    wavelengths = np.linspace(1500, 2000, 100)  # nm
    T = 20  # °C
    H = 30  # %
    P = 99000  # Pa
    
    # Calculate refractive index at different wavelengths
    n_ciddor = np.array([ciddor(wave=w, t=T, p=P, rh=H) for w in wavelengths])
    n_edlen = np.array([edlen(wave=w, t=T, p=P, rh=H) for w in wavelengths])
    n_mathar = np.array([n(w/1000, 273.15+T, P, H) for w in wavelengths])
    
    # Calculate wavelength derivatives
    dn_dlambda_ciddor = np.gradient(n_ciddor, wavelengths)
    dn_dlambda_edlen = np.gradient(n_edlen, wavelengths)
    dn_dlambda_mathar = np.gradient(n_mathar, wavelengths)
    
    return {
        'wavelengths': wavelengths,
        'n_ciddor': n_ciddor,
        'n_edlen': n_edlen,
        'n_mathar': n_mathar,
        'dn_dlambda_ciddor': dn_dlambda_ciddor,
        'dn_dlambda_edlen': dn_dlambda_edlen,
        'dn_dlambda_mathar': dn_dlambda_mathar
    }

### 10. Uncertainty Propagation
**Purpose**: Propagates coefficient uncertainties to provide confidence intervals for refractive index predictions.


In [7]:
def propagate_uncertainty(results_json, T, H, P):
    """
    Propagate coefficient uncertainties to refractive index predictions
    
    Parameters:
    -----------
    results_json : dict
        Regression results with uncertainties
    T, H, P : float or array
        Environmental conditions
        
    Returns:
    --------
    dict
        Uncertainty estimates for refractive index predictions
    """
    coeff = results_json['coefficients']
    errors = results_json['standard_errors']
    
    # Calculate refractive index
    n_pred = coeff['const'] + coeff['temperature']*T + coeff['humidity']*H + coeff['pressure']*P
    
    # Propagate uncertainties (assuming independent errors)
    var_n = (errors['const']**2 + 
             (errors['temperature']*T)**2 + 
             (errors['humidity']*H)**2 + 
             (errors['pressure']*P)**2)
    
    std_n = np.sqrt(var_n)
    
    return {
        'n_pred': n_pred,
        'std_n': std_n,
        'relative_uncertainty': std_n / n_pred * 1e6,  # ppm
        'expanded_uncertainty_k2': 2 * std_n
    }

## Complete Analysis Workflow

In [9]:
def run_complete_validation():
    """
    Complete model validation workflow
    """
    print("="*60)
    print("MODEL VALIDATION AND COMPARISON ANALYSIS")
    print("="*60)
    
    # 1. Load experimental results
    results_json = load_regression_results('regression_results.json')
    n_1762 = refractive_function(results_json)
    
    # 2. Generate comparison plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    plt.subplots_adjust(wspace=0.3)
    
    # Temperature comparison
    pressure = 99000
    humidity = 30
    temperature = np.linspace(15, 35, 1000)
    
    models_temp = {
        'Ciddor@1700nm': np.array([ciddor(wave=1762, t=i, p=pressure, rh=humidity) 
                                   for i in temperature]),
        'Edlen@1700nm': np.array([edlen(wave=1762, t=i, p=pressure, rh=humidity) 
                                  for i in temperature]),
        'Mathar@1762nm': n(1.762, 273.15+temperature, pressure, humidity),
        'This Work@1762nm': n_1762(temperature, humidity, pressure)
    }
    
    # Plot all comparisons
    for (label, values), color in zip(models_temp.items(), colorlist):
        axes[0].plot(temperature, values, color=color, label=label)
    
    # Similar code for humidity and pressure comparisons...
    
    # 3. Calculate model deviations
    deviations = calculate_model_deviations(models_temp, models_humidity, models_pressure)
    
    print("\nModel Deviation Analysis (relative to This Work):")
    print("-"*50)
    for model_name, stats in deviations.items():
        print(f"\n{model_name}:")
        print(f"  Temperature RMSE: {stats['temperature_rmse_ppm']:.1f} ppm")
        print(f"  Humidity RMSE: {stats['humidity_rmse_ppm']:.1f} ppm")
        print(f"  Pressure RMSE: {stats['pressure_rmse_ppm']:.1f} ppm")
    
    # 4. Enhanced sensitivity analysis
    sensitivity_results = analyze_enhanced_sensitivity(models_humidity)
    
    print("\nEnhanced Humidity Sensitivity Analysis:")
    print("-"*50)
    print(f"Mathar model slope: {sensitivity_results['mathar_slope']:.3e} per %")
    print(f"This Work slope: {sensitivity_results['this_work_slope']:.3e} per %")
    print(f"Enhancement: {sensitivity_results['enhancement_percent']:.1f}%")
    
    # 5. Uncertainty propagation
    uncertainty = propagate_uncertainty(results_json, T=20, H=30, P=99000)
    
    print("\nUncertainty Analysis:")
    print("-"*50)
    print(f"Predicted n: {uncertainty['n_pred']:.8f}")
    print(f"Standard uncertainty: {uncertainty['std_n']:.2e}")
    print(f"Relative uncertainty: {uncertainty['relative_uncertainty']:.1f} ppm")
    print(f"Expanded uncertainty (k=2): {uncertainty['expanded_uncertainty_k2']:.2e}")
    
    # 6. Wavelength dependence analysis
    wavelength_results = analyze_wavelength_dependence()
    
    # Create wavelength dependence plot
    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    ax1.plot(wavelength_results['wavelengths'], wavelength_results['n_ciddor'], 
             label='Ciddor', color=pub_purple)
    ax1.plot(wavelength_results['wavelengths'], wavelength_results['n_edlen'], 
             label='Edlén', color=pub_teal)
    ax1.plot(wavelength_results['wavelengths'], wavelength_results['n_mathar'], 
             label='Mathar', color=pub_gold)
    ax1.axvline(x=1762, color=pub_red, linestyle='--', label='1762 nm')
    ax1.set_xlabel('Wavelength (nm)')
    ax1.set_ylabel('Refractive Index')
    ax1.set_title('Wavelength Dependence of Refractive Index Models')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(wavelength_results['wavelengths'], wavelength_results['dn_dlambda_ciddor'], 
             label='Ciddor', color=pub_purple)
    ax2.plot(wavelength_results['wavelengths'], wavelength_results['dn_dlambda_edlen'], 
             label='Edlén', color=pub_teal)
    ax2.plot(wavelength_results['wavelengths'], wavelength_results['dn_dlambda_mathar'], 
             label='Mathar', color=pub_gold)
    ax2.axvline(x=1762, color=pub_red, linestyle='--')
    ax2.set_xlabel('Wavelength (nm)')
    ax2.set_ylabel('dn/dλ (nm⁻¹)')
    ax2.set_title('Dispersion (dn/dλ) of Refractive Index Models')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'deviations': deviations,
        'sensitivity_results': sensitivity_results,
        'uncertainty': uncertainty,
        'wavelength_results': wavelength_results
    }

## Key Results and Interpretation

### 1. **Coefficient Comparison**

| Model | $\alpha_T$ (per °C) | $\alpha_H$ (per %) | $\alpha_P$ (per Pa) |
|-------|---------------------|-------------------|-------------------|
| **This Work** | $-8.8474 \times 10^{-7}$ | $-1.3152 \times 10^{-8}$ | $+2.5950 \times 10^{-7}$ |
| **Mathar@1762nm** | $-8.85 \times 10^{-7}$ | $-1.122 \times 10^{-8}$ | $+2.60 \times 10^{-7}$ |
| **Ciddor@1700nm** | $-8.82 \times 10^{-7}$ | $-1.10 \times 10^{-8}$ | $+2.58 \times 10^{-7}$ |
| **Edlén@1700nm** | $-8.80 \times 10^{-7}$ | $-1.08 \times 10^{-8}$ | $+2.57 \times 10^{-7}$ |

### 2. **Enhanced Humidity Sensitivity**

- **This Work**: $-1.3152 \times 10^{-8}$ per %
- **Mathar model**: $-1.122 \times 10^{-8}$ per %
- **Enhancement**: $17.2\%$ relative to Mathar model
- **Physical origin**: Proximity to water vapor absorption lines at 1761.04 nm and 1762.49 nm

### 3. **Model Deviations**

- **Temperature agreement**: All models within 0.3% of This Work
- **Humidity deviation**: Ciddor and Edlén underestimate by 16-18%
- **Pressure agreement**: All models within 0.8% of This Work
- **Wavelength specificity**: 1700 nm models show significant deviations at 1762 nm

### 4. **Uncertainty Analysis**

- **Standard uncertainty**: $2.12 \times 10^{-7}$ (0.2 ppb)
- **Expanded uncertainty (k=2)**: $4.24 \times 10^{-7}$ (0.4 ppb)
- **Dominant uncertainty source**: Environmental sensor calibration
- **Quantum reference stability**: $\delta n < 10^{-12}$ (three orders better)

### 5. **Practical Implications**

1. **Wavelength specificity matters**: 62 nm difference (1700→1762 nm) causes 17% humidity coefficient change
2. **Model selection criteria**: 
   - Use Mathar model for 1762 nm applications
   - This Work provides highest precision for quantum networks
   - Ciddor/Edlén suitable for general 1700 nm applications
3. **Measurement hierarchy**: Environmental sensors limit precision, not quantum reference

## Physical Interpretation

The enhanced humidity sensitivity at 1762 nm results from:

1. **Spectral proximity**: 1762 nm lies between water vapor absorption lines at 1761.04 nm and 1762.49 nm
2. **Kramers-Kronig causality**: Absorption features create enhanced dispersion
3. **Wavelength-dependent Sellmeier coefficients**: Water vapor dispersion varies nonlinearly with wavelength
4. **Temperature-pressure coupling**: Environmental parameters interact in complex ways

## Usage Recommendations

### For Quantum Network Applications:
```python
# Use This Work model for highest precision at 1762 nm
n_pred = n_1762(T=25.0, H=35.0, P=99000)
uncertainty = 4.24e-7  # Expanded uncertainty (k=2)
```

### For General 1762 nm Applications:
```python
# Use Mathar model with wavelength-specific coefficients
from models.mathar.Mathar2007 import n
n_pred = n(1.762, 273.15+25.0, 99000, 35.0)
```

### For 1700 nm Applications:
```python
# Use Ciddor or Edlén models
from models.nist.refractive_index import ciddor
n_pred = ciddor(wave=1700, t=25.0, p=99000, rh=35.0)
```

This comprehensive validation demonstrates the importance of wavelength-specific refractive index measurements and provides uncertainty-quantified models for precision applications in quantum networking and atmospheric sensing.